# Source Scenes for Static Layers over Australia

- The following notebook is used to search for the unique set of scenes that can be used to produced burst static layer products for the Australian region.
- The scenes used to process the complete set of static layers over Australia are the same as the NASA JPL team used to create their RTC_S1_STATIC product.
- These can be viewed using the ASF Search UI and filtering by the OPERA-S1 dataset and RTC-STATIC file type
- E.g. https://search.asf.alaska.edu/#/?zoom=3.000&center=-5.508,-13.732&dataset=OPERA-S1&productTypes=CSLC-STATIC.



# Imports

In [ ]:
# initial setup
import os
import asf_search as asf
from eof.download import download_eofs
from datetime import datetime
import geopandas as gpd
from shapely.geometry import shape

## Download our Australian region of interest and convert to wkt

In [ ]:
url = 'https://deant-data-public-dev.s3.ap-southeast-2.amazonaws.com/persistent/aus_aoi_unioned.geojson'
gdf = gpd.read_file(url)
gdf['geometry_wkt'] = gdf['geometry'].apply(lambda geom: geom.wkt)
wkt = gdf['geometry_wkt'].values[0]
wkt
gdf.plot()

# search for opera rtc-s1 static products

In [ ]:

results = asf.search(platform=[asf.PLATFORM.SENTINEL1], 
                     intersectsWith=wkt, 
                     maxResults=15_000, 
                     processingLevel='RTC-STATIC',
                     beamMode='IW',
                    )
print(len(results))

## Key metadata into geopandas df

In [ ]:
rows = []
for i,r in enumerate(results):
    burst_id = r.umm["AdditionalAttributes"][5]["Values"][0].lower()
    scene_id = r.umm["InputGranules"][0]

    scene_dt = datetime.strptime(
        scene_id.split("_")[6], "%Y%m%dT%H%M%S"
    )

    geom = shape(r.geometry)  # convert GeoJSON → shapely

    rows.append(
        {
            "burst_id": burst_id,
            "scene_id": scene_id,
            "scene_dt": scene_dt,
            "geometry": geom,
        }
    )
    # if i == 1000:
    #     break

gdf = gpd.GeoDataFrame(
    rows,
    geometry="geometry",
    crs="EPSG:4326",  # CMR geometries are lon/lat
)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Ensure datetime
gdf['scene_dt'] = pd.to_datetime(gdf['scene_dt'])

# Group by scene_id to remove duplicate bursts in same scene
scene_counts = gdf.groupby('scene_id').first().reset_index()

# Set datetime index for resampling
scene_counts.set_index('scene_dt', inplace=True)

# Resample monthly
monthly_counts = scene_counts['scene_id'].resample('M').count()

# Plot
fig, ax = plt.subplots(figsize=(10,5))
monthly_counts.plot(kind='bar', width=0.8, ax=ax)

# Major ticks: every 12 months → show year
ax.set_xticks(range(0, len(monthly_counts), 12))
ax.set_xticklabels([d.strftime('%Y') for d in monthly_counts.index[::12]], rotation=45)

ax.set_xlabel("Date")
ax.set_ylabel("Number of Scenes per Month")
ax.set_title(f"Monthly Count of Input Scenes Used to Create Static Layers over Australia (total = {len(scene_counts)})")
ax.grid(axis='x', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()


In [ ]:
gdf.plot()

In [ ]:
gdf.to_file("australia_static_layer_source_scenes.geojson", driver="GeoJSON")  # readable, portable

In [ ]:
scene_counts.sort_values('scene_dt',ascending=False)

### Add additonal scenes and save to file

In [ ]:
# These are additional scenes that were identified in the ongoing pipeline with missing bursts
additional_scenes = [
    'S1C_IW_SLC__1SDV_20260420T201107_20260420T201134_007304_00ECF1_B731',
    'S1C_IW_SLC__1SDV_20260419T210955_20260419T211031_007290_00EC7B_6E5D',
    'S1C_IW_SLC__1SDV_20260420T214727_20260420T214758_007305_00ECFA_DD8D',
    'S1C_IW_SLC__1SDV_20260420T214935_20260420T214958_007305_00ECFA_2D73',
]

In [ ]:
australia_static_layer_source_scenes = scene_counts.sort_values('scene_dt',ascending=True).scene_id.values
with open("australia_static_layer_source_scene_ids.txt", "w") as f:
    for scene in list(australia_static_layer_source_scenes)+additional_scenes:
        f.write(f"{scene}\n")

## Upload Files
- Files have been uploaded here - https://data.dev.dea.ga.gov.au/?prefix=projects/s1_nrb/production_static_layer_scene_lists/